# 03b · RoBERTa penultimate embeddings — a richer sentiment representation

**Why both the score vector and the embedding come from the same place.** The 3-D `[pos,neg,neu]` score
is the model's *final sentiment decision*; the 768-d embedding is the encoder representation the model
*built in order to make that decision*. Because Twitter-RoBERTa is **fine-tuned for sentiment**, that
representation is organized around sentiment — it is not a general semantic embedding, it is a
sentiment-shaped one. So it is a legitimate sentiment representation, just richer than the score.

**When the score vector makes more sense:** when the claim is specifically about *valence/sentiment* —
it is the purest, most interpretable object, and it is exactly what Step 1 validated against behavior
(cv-R² 0.34). Use it as the headline.

**When the embedding makes more sense:** when you need a richer, higher-**reliability** representation
comparable to Jin's 512-d USE vectors — the 3-D score's run-split reliability was only ~0.03, which is
what capped the 04b IS-RSA. The embedding may lift you off that noise floor. *Caveat:* it still encodes
the content that predicts sentiment, so a brain effect reads as 'tracks the sentiment representation,'
slightly broader than 'tracks valence.' Report it as the reliability-motivated comparison, not a replacement.

This notebook (1) extracts the embeddings, (2) builds the same-format similarity matrices, (3) checks
whether reliability actually improves over the 3-D score.

> [!warning] Reliability is representation- AND split-dependent (added 2026-07-21). The embedding's *measure* reliability is high (behavioural cv-R²=0.443, §1.10), but its *similarity-structure* (RDM) run-split reliability here is ~0.005 — not above the 3-D score's ~0.03. So the embedding is the better-validated **measure**, not necessarily the more reliable **RDM** for IS-RSA. State the level explicitly.


In [2]:
import pandas as pd, numpy as np, torch, pickle, os
from transformers import AutoTokenizer, AutoModel
MODEL_ID = "cardiffnlp/twitter-roberta-base-sentiment-latest"   # SAME model as Step 0 winner
tok = AutoTokenizer.from_pretrained(MODEL_ID)
enc = AutoModel.from_pretrained(MODEL_ID)   # base encoder (fine-tuned weights), no classification head
enc.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"; enc.to(device)

sc = pd.read_csv("results/scored/00__reflection_sentiment.csv")
sc["Character"] = sc["Character"].str.lower().str.strip()
meta = sc.dropna(subset=["Raw_Text"]).drop_duplicates(["Participant","Run","Character"])
print(len(meta), "reflections to embed")

@torch.no_grad()
def embed(text):
    e = tok(str(text), return_tensors="pt", truncation=True, max_length=512).to(device)
    h = enc(**e).last_hidden_state[0]                 # (tokens, 768) penultimate activations
    m = e["attention_mask"][0].unsqueeze(-1)
    return ((h*m).sum(0) / m.sum().clamp(min=1)).cpu().numpy()   # attention-masked mean pool

embs = {}
for k,(_,r) in enumerate(meta.iterrows()):
    embs[(r.Participant, int(r.Run), r.Character)] = embed(r.Raw_Text)
    if k % 200 == 0: print(f"  {k}/{len(meta)}")
os.makedirs("results/embeddings", exist_ok=True)
pickle.dump(embs, open("results/embeddings/03b__roberta_embeddings.pkl","wb"))
print("saved", len(embs), "embeddings (768-d)")

/Users/rheamadhogarhia/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1272 reflections to embed
  0/1272
  200/1272
  400/1272
  600/1272
  800/1272
  1000/1272
  1200/1272
saved 1272 embeddings (768-d)


## Build between-subject similarity (same format as `03__sentiment_sim_byrun_bychar.npy`) + reliability check

In [3]:
# = Jin step03 cosine_similarity, imported VERBATIM (jin_code/jin_step03.py).
#   Verified identical to the previous local cos() on 5000 vectors (max|diff|=0) and on the
#   zero-norm / NaN edge cases. His version additionally returns NaN for a missing embedding
#   where the local one raised ValueError.
import sys; sys.path.insert(0, 'jin_code')
from jin_step03 import cosine_similarity
import numpy as np, json as _json, pickle
from scipy.stats import spearmanr
emb = pickle.load(open("results/embeddings/03b__roberta_embeddings.pkl","rb"))
overlap = _json.load(open("results/jinrep/03__isrsa_subject_order.json"))   # 29 overlap, Jin order
CHAR = ["jack","kate","randall","kevin"]
sim = {}
for g in [1,2,3]:
    subs = overlap[str(g)]
    for run in range(1,11):
        chs = []
        for ch in CHAR:
            v = [emb.get((s,run,ch)) for s in subs]
            chs.append([cosine_similarity(v[i],v[j]) if (v[i] is not None and v[j] is not None) else np.nan
                         for i in range(len(v)) for j in range(i+1,len(v))])
        sim[g,run] = np.array(chs)                      # (4 char, n_pairs) — 29-overlap, Jin order
np.save("results/jinrep/03b__roberta_embed_sim_byrun_bychar.npy", sim, allow_pickle=True)
print("saved results/jinrep/03b__roberta_embed_sim_byrun_bychar.npy (same format as the 3-D sentiment file)")

# reliability: odd vs even runs split-half of the RDM (same metric that gave the 3-D score ~0.03)
odd  = np.concatenate([sim[g,r].ravel() for g in [1,2,3] for r in [1,3,5,7,9]])
even = np.concatenate([sim[g,r].ravel() for g in [1,2,3] for r in [2,4,6,8,10]])
m = ~(np.isnan(odd)|np.isnan(even))
print(f"embedding RDM split-half (odd vs even runs): {spearmanr(odd[m],even[m])[0]:.3f}   (3-D score was ~0.03)")

saved results/jinrep/03b__roberta_embed_sim_byrun_bychar.npy (same format as the 3-D sentiment file)
embedding RDM split-half (odd vs even runs): 0.005   (3-D score was ~0.03)


## Run the IS-RSA on the embedding

In `04b`, point the sentiment-similarity loader at this file instead of the 3-D one:

```python
SENT = np.load("results/jinrep/03b__roberta_embed_sim_byrun_bychar.npy", allow_pickle=True).item()
```

then re-run `isrsa()` and the dual-readout cell. Compare its regions (and the reliability above) to the
3-D score run — if the embedding clears the reliability floor and finds regions the 3-D score didn't,
that's the reliability story; if it just recapitulates Jin's USE result, the Step-3 RSA (USE vs RoBERTa)
will have already told you they share geometry.

**Done in `04b.1`:** the embedding IS-RSA recovers before-movie ROIs [9, 60] (q<0.01) and [9, 60, 78] Fisher-combined, where the 3-D score (`04b`) was fully null — a modest reliability lift, consistent with the weak USE<->RoBERTa geometry agreement (rho ~ 0.08).

## 3b.4 · Decode `like` from the embeddings (mech-interp)  *(moved from exploratory 09)*

Does the 768-d penultimate encode viewer stance even though the 3-D head discards it? LOGO ridge from end-state embedding → survey `like` vs `positive_emotion`, with 3-D valence→like as baseline.

In [4]:
import pickle, numpy as np, pandas as pd, scipy.io as sio
from pathlib import Path
from collections import defaultdict
from scipy.stats import spearmanr
from config import SURVEY_DIR
CHARS=["jack","kate","randall","kevin"]
def _nrm(x): return "".join(c for c in str(x) if c.isdigit())
emb=pickle.load(open("results/embeddings/03b__roberta_embeddings.pkl","rb"))
acc=defaultdict(list)
for (p,run,ch),v in emb.items(): acc[(_nrm(p),ch)].append(np.asarray(v,float))
E={k:np.mean(v,0) for k,v in acc.items()}                         # end-state embedding per (nid,char)
like={}; pos={}
for f in sorted(Path(SURVEY_DIR).glob("*.mat")):
    if f.name=="labels.mat": continue
    m=sio.loadmat(f)
    if not all(f"data{i}" in m for i in range(1,7)): continue
    nid=_nrm(f.stem)
    for ci,ch in enumerate(CHARS):
        like[(nid,ch)]=float(np.asarray(m["data3"],float)[2,ci])   # data3 idx2 = "like"
        pos[(nid,ch)] =float(np.asarray(m["data2"],float)[1,ci])   # data2 idx1 = "positive emotion"
b=pd.read_csv("results/baselines/00__character_vectors_simple_Twitter_RoB.csv")
b["ch"]=b["Character"].str.lower().str.strip(); b["v"]=b["positive"]-b["negative"]; b["nid"]=b["Participant"].map(_nrm)
val=b[b["Run"].between(1,10)].groupby(["nid","ch"])["v"].mean().to_dict()
keys=[k for k in E if k in like and not np.isnan(like[k])]
X=np.vstack([E[k] for k in keys]); yl=np.array([like[k] for k in keys]); yp=np.array([pos[k] for k in keys])
vbase=np.array([val.get(k,np.nan) for k in keys]); groups=np.array([int(k[0][0]) for k in keys])
def logo_ridge(X,y,groups,lam=25.0):
    pred=np.full(len(y),np.nan)
    for g in np.unique(groups):
        tr=groups!=g; te=groups==g; mu=X[tr].mean(0); sd=X[tr].std(0)+1e-8
        Xtr=(X[tr]-mu)/sd; Xte=(X[te]-mu)/sd; ym=y[tr].mean()
        w=np.linalg.solve(Xtr.T@Xtr+lam*np.eye(Xtr.shape[1]), Xtr.T@(y[tr]-ym)); pred[te]=Xte@w+ym
    return 1-np.nansum((y-pred)**2)/np.nansum((y-y.mean())**2), spearmanr(y,pred)[0]
print(f"n samples={len(keys)}  X={X.shape}")
for name,y in [("like",yl),("positive_emotion",yp)]:
    r2,rho=logo_ridge(X,y,groups); print(f"  embedding -> {name:16s}: LOGO CV-R2={r2:+.3f}  Spearman(pred,actual)={rho:+.3f}")
mm=~np.isnan(vbase); print(f"  3-D valence -> like (baseline): Spearman={spearmanr(yl[mm],vbase[mm])[0]:+.3f}")

n samples=124  X=(124, 768)
  embedding -> like            : LOGO CV-R2=-0.524  Spearman(pred,actual)=+0.284
  embedding -> positive_emotion: LOGO CV-R2=-0.504  Spearman(pred,actual)=+0.304
  3-D valence -> like (baseline): Spearman=+0.385


/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_10878/549822415.py:32: RuntimeWarning: divide by zero encountered in matmul
  w=np.linalg.solve(Xtr.T@Xtr+lam*np.eye(Xtr.shape[1]), Xtr.T@(y[tr]-ym)); pred[te]=Xte@w+ym
/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_10878/549822415.py:32: RuntimeWarning: overflow encountered in matmul
  w=np.linalg.solve(Xtr.T@Xtr+lam*np.eye(Xtr.shape[1]), Xtr.T@(y[tr]-ym)); pred[te]=Xte@w+ym
/var/folders/lk/34qrvdj11cq5ln8vscwdnrg00000gn/T/ipykernel_10878/549822415.py:32: RuntimeWarning: invalid value encountered in matmul
  w=np.linalg.solve(Xtr.T@Xtr+lam*np.eye(Xtr.shape[1]), Xtr.T@(y[tr]-ym)); pred[te]=Xte@w+ym


> [!warning] Null-so-far: embedding predicts `like` (ρ≈0.28) and `positive_emotion` (0.30) about equally, and *worse* than plain 3-D valence predicts `like` (0.385); CV-R² is negative (768-d overfits 124 samples). No evidence of a hidden, selective `like` code — redo with PCA-reduction / tuned λ before concluding.

## 3b.5 · Extract all layers for layer-wise brain–model RSA  *(Direction 5, feeds 04b.1)*

In [5]:
# 3b.5 · Extract ALL 13 hidden layers (for layer-wise brain-model RSA in 04b.1). RUN LOCALLY (needs the model).
# Reuses tok/enc/meta from this notebook -- run 03b's extraction cell first.
import torch, numpy as np, pickle
enc.config.output_hidden_states = True
def embed_all_layers(text):
    e = tok(str(text), return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad(): hs = enc(**e).hidden_states          # tuple(13) each (1, tokens, 768)
    return np.stack([h[0].mean(0).numpy() for h in hs])        # (13, 768) mean-pooled per layer
layer_emb = {(r.Participant, int(r.Run), r.Character): embed_all_layers(r.Raw_Text) for _, r in meta.iterrows()}
pickle.dump(layer_emb, open("results/embeddings/03b__roberta_layerwise.pkl", "wb"))
print("saved", len(layer_emb), "x (13,768) layer-wise embeddings -> results/embeddings/03b__roberta_layerwise.pkl")

saved 1272 x (13,768) layer-wise embeddings -> results/embeddings/03b__roberta_layerwise.pkl
